# EDA — Healthcare Provider Fraud Detection

Análisis exploratorio del dataset `rohitrox/healthcare-provider-fraud-detection-analysis`.  
Objetivo: entender la estructura de los datos, el desbalance de clases y los patrones de fraude.

In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from healthcare_fraud.data import clean_dataframe, load_dataset, validate_dataframe

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 1. Carga de datos

In [ ]:
# Carga y validación de todas las tablas
dfs_raw = load_dataset()
print("Tablas disponibles:", list(dfs_raw.keys()))

In [ ]:
# Validar y limpiar cada tabla
dfs = {}
for name, df in dfs_raw.items():
    validated = validate_dataframe(df, name)
    dfs[name] = clean_dataframe(validated, name)

# Resumen de dimensiones
summary = pd.DataFrame(
    [(k, *v.shape) for k, v in dfs.items()], columns=["tabla", "filas", "columnas"]
)
summary

## 2. Estructura de cada tabla

In [ ]:
for name, df in dfs.items():
    print(f"\n### {name} ###")
    display(df.dtypes.to_frame("dtype").T)
    display(df.head(2))

## 3. Desbalance de clases (variable objetivo)

In [ ]:
labels = dfs.get("labels_train")
if labels is not None and "PotentialFraud" in labels.columns:
    counts = labels["PotentialFraud"].value_counts()
    ratio = counts[1] / len(labels) * 100
    print(f"Proveedores totales: {len(labels):,}")
    print(f"Fraude (1): {counts.get(1, 0):,}  |  No fraude (0): {counts.get(0, 0):,}")
    print(f"Ratio de fraude: {ratio:.2f}%")

    fig, ax = plt.subplots(figsize=(5, 4))
    counts.rename({1: "Fraude", 0: "No fraude"}).plot.bar(ax=ax, color=["#e74c3c", "#2ecc71"])
    ax.set_title("Distribución de la variable objetivo")
    ax.set_xlabel("")
    ax.set_ylabel("Proveedores")
    plt.tight_layout()
    plt.show()

## 4. Distribuciones de montos de reclamación

In [ ]:
amount_col = "InscClaimAmtReimbursed"

for table_name in ("inpatient", "outpatient"):
    df = dfs.get(table_name)
    if df is None or amount_col not in df.columns:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df[amount_col].dropna().plot.hist(bins=50, ax=axes[0], color="steelblue")
    axes[0].set_title(f"{table_name} — distribución {amount_col}")
    axes[0].set_xlabel("USD")

    df[amount_col].dropna().plot.box(ax=axes[1])
    axes[1].set_title(f"{table_name} — boxplot {amount_col}")
    plt.tight_layout()
    plt.show()

    print(df[amount_col].describe().to_string())

## 5. Análisis de nulos por columna

In [ ]:
for name, df in dfs.items():
    null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        print(f"{name}: sin nulos")
        continue

    fig, ax = plt.subplots(figsize=(10, max(3, len(null_pct) * 0.4)))
    null_pct.plot.barh(ax=ax, color="salmon")
    ax.set_title(f"{name} — % nulos por columna")
    ax.set_xlabel("% nulos")
    plt.tight_layout()
    plt.show()

## 6. Heatmap de correlaciones (inpatient)

In [ ]:
df_ip = dfs.get("inpatient")
if df_ip is not None:
    numeric_cols = df_ip.select_dtypes(include="number").columns.tolist()
    if len(numeric_cols) > 1:
        corr = df_ip[numeric_cols].corr()
        fig, ax = plt.subplots(figsize=(min(14, len(numeric_cols)), min(12, len(numeric_cols))))
        sns.heatmap(corr, annot=False, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
        ax.set_title("Correlaciones — inpatient")
        plt.tight_layout()
        plt.show()

## 7. Distribución geográfica (beneficiary)

In [ ]:
bene = dfs.get("beneficiary")
if bene is not None and "State" in bene.columns:
    top_states = bene["State"].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(12, 5))
    top_states.plot.bar(ax=ax, color="cornflowerblue")
    ax.set_title("Top 20 estados por número de beneficiarios")
    ax.set_xlabel("State code")
    ax.set_ylabel("Beneficiarios")
    plt.tight_layout()
    plt.show()

## 8. Conclusiones

**Completar después de ejecutar el notebook con datos reales:**

- **Desbalance de clases**: ratio de fraude aproximado ~10% (confirmar con celda 3)
- **Tablas principales**: beneficiary (demografía), inpatient/outpatient (reclamaciones), labels_train (etiquetas por proveedor)
- **Variables discriminantes**: montos de reclamación, número de reclamaciones por proveedor, condiciones crónicas acumuladas
- **Calidad de datos**: columnas de diagnóstico secundario y procedimientos tienen alta tasa de nulos — candidatos a ingeniería de features agregadas
- **Siguiente paso**: construir features a nivel proveedor (Fase 02) agregando inpatient + outpatient + beneficiary